In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil
datasetPath = "/content/drive/MyDrive/dataset"
localDataset = "/content/dataset"
shutil.copytree(datasetPath, localDataset)


'/content/dataset'

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from tqdm import tqdm
import numpy as np
import os
import joblib
from sklearn.neighbors import NearestNeighbors
import json
import yaml
from PIL import ImageFile

In [ ]:
def load_config():
    with open('params.yaml', 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    return config

config = load_config()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_path = 'dataset'
print(f"Data path: {data_path}")

Data path: dataset


In [ ]:
# Для обучения
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# Для теста
val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
full_dataset = ImageFolder(data_path, transform=train_transform)
val_dataset = ImageFolder(data_path, transform=val_transform)

num_classes = len(full_dataset.classes)
print(f"Number of classes: {num_classes}")
print(f"Classes: {full_dataset.classes}")

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

Number of classes: 6
Classes: ['Матмех', 'НИИФ', 'ПМ-ПУ', 'Физфак', 'Химфак', 'Шайба']


In [ ]:
batch_size = config['training']['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class PlaceRecognizerModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = models.resnet50(pretrained=True)

        for param in self.model.parameters():
            param.requires_grad = False

        in_features = self.model.fc.in_features

        self.model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.model(x)

model = PlaceRecognizerModel(num_classes=num_classes)
model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 120MB/s]


PlaceRecognizerModel(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(

In [ ]:
optimizer = optim.AdamW(
    model.model.fc.parameters(),
    lr=config['training']['learning_rate'],
    weight_decay=config['training']['weight_decay']
)

In [ ]:
num_epochs = config['training']['epochs']
best_accuracy = 0

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.001,
    epochs=num_epochs,
    steps_per_epoch=len(train_loader)
)

In [ ]:
for epoch in range(num_epochs):
    model.train()
    for (idx, (x, y)) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        model_result = model(x)
        loss = torch.nn.functional.cross_entropy(model_result, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        print(f'Batch: #{epoch}/{idx}, loss: {loss:.2f}')

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

    accuracy = 100 * correct / total
    print(f'Epoch {epoch} completed. Accuracy: {accuracy:.2f}%')

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'accuracy': accuracy,
            'class_names': full_dataset.classes,
            'num_classes': num_classes
        }, 'bestModel.pth')
        print(f'Model saved! Best accuracy: {accuracy:.2f}%')

print("\nTraining completed!")
print(f"Best accuracy achieved: {best_accuracy:.2f}%")

torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': full_dataset.classes,
    'num_classes': num_classes,
}, 'finalModel.pth')
print("Final model saved as 'finalModel.pth'")

Batch: #0/0, loss: 1.98
Batch: #0/1, loss: 1.72
Batch: #0/2, loss: 2.04
Batch: #0/3, loss: 2.11
Batch: #0/4, loss: 1.87
Batch: #0/5, loss: 1.76
Batch: #0/6, loss: 1.76
Batch: #0/7, loss: 1.84
Batch: #0/8, loss: 1.76
Batch: #0/9, loss: 1.79
Batch: #0/10, loss: 1.76
Batch: #0/11, loss: 1.65
Batch: #0/12, loss: 1.66
Batch: #0/13, loss: 1.81
Batch: #0/14, loss: 2.05
Batch: #0/15, loss: 1.83
Batch: #0/16, loss: 1.92
Batch: #0/17, loss: 1.92
Batch: #0/18, loss: 1.84
Batch: #0/19, loss: 1.78
Batch: #0/20, loss: 1.59
Batch: #0/21, loss: 1.92
Batch: #0/22, loss: 1.88
Batch: #0/23, loss: 1.57
Batch: #0/24, loss: 1.63
Batch: #0/25, loss: 1.65
Batch: #0/26, loss: 1.53
Batch: #0/27, loss: 1.43
Batch: #0/28, loss: 1.57
Batch: #0/29, loss: 1.53
Batch: #0/30, loss: 1.52
Batch: #0/31, loss: 1.53
Batch: #0/32, loss: 1.53
Batch: #0/33, loss: 1.27
Batch: #0/34, loss: 1.61
Batch: #0/35, loss: 1.48
Batch: #0/36, loss: 1.44
Batch: #0/37, loss: 1.23
Batch: #0/38, loss: 1.53
Batch: #0/39, loss: 1.33
Batch: #0/

Обучение успешно прошло на 5 эпохах. Дообучаем до 10 эпох.

In [ ]:
checkpoint = torch.load('bestModel.pth')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1

In [ ]:
for epoch in range(start_epoch + 1, num_epochs):
    model.train()
    for (idx, (x, y)) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        model_result = model(x)
        loss = torch.nn.functional.cross_entropy(model_result, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        print(f'Batch: #{epoch}/{idx}, loss: {loss:.2f}')

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

    accuracy = 100 * correct / total
    print(f'Epoch {epoch} completed. Accuracy: {accuracy:.2f}%')

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'accuracy': accuracy,
            'class_names': full_dataset.classes,
            'num_classes': num_classes
        }, 'bestModel.pth')
        print(f'Model saved! Best accuracy: {accuracy:.2f}%')

print("\nTraining completed!")
print(f"Best accuracy achieved: {best_accuracy:.2f}%")

torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': full_dataset.classes,
    'num_classes': num_classes,
}, 'finalModel.pth')
print("Final model saved as 'finalModel.pth'")

Batch: #5/0, loss: 0.73
Batch: #5/1, loss: 0.81
Batch: #5/2, loss: 0.84
Batch: #5/3, loss: 0.42
Batch: #5/4, loss: 0.68
Batch: #5/5, loss: 0.68
Batch: #5/6, loss: 0.53
Batch: #5/7, loss: 0.68
Batch: #5/8, loss: 0.58
Batch: #5/9, loss: 0.60
Batch: #5/10, loss: 0.60
Batch: #5/11, loss: 0.90
Batch: #5/12, loss: 0.58
Batch: #5/13, loss: 0.60
Batch: #5/14, loss: 0.69
Batch: #5/15, loss: 0.45
Batch: #5/16, loss: 0.65
Batch: #5/17, loss: 0.49
Batch: #5/18, loss: 0.51
Batch: #5/19, loss: 0.50
Batch: #5/20, loss: 0.82
Batch: #5/21, loss: 0.52
Batch: #5/22, loss: 0.66
Batch: #5/23, loss: 0.80
Batch: #5/24, loss: 0.42
Batch: #5/25, loss: 0.54
Batch: #5/26, loss: 0.67
Batch: #5/27, loss: 0.67
Batch: #5/28, loss: 0.82
Batch: #5/29, loss: 0.70
Batch: #5/30, loss: 0.39
Batch: #5/31, loss: 0.48
Batch: #5/32, loss: 0.52
Batch: #5/33, loss: 0.89
Batch: #5/34, loss: 0.72
Batch: #5/35, loss: 0.75
Batch: #5/36, loss: 0.49
Batch: #5/37, loss: 0.85
Batch: #5/38, loss: 0.73
Batch: #5/39, loss: 0.43
Batch: #5/